# Cold-start QPP dose-response: OFFLINE (batch retrain), training-set *size*


In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "config.py").exists() and (p / "qpp").is_dir())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from qpp.config import PARSED_RESULTS_ROOT, METRIC, XCOL, YCOL, SEED
from qpp.models import set_seed, TrinoGraphGINPlanEncoder
from qpp.qpp_harness import (
    subset_bundle,
    nested_query_subsamples,
    training_subset_stats,
    build_mixed_bundle_lenient,
    build_point_dataset,
    filter_embeddings_by_qids,
    fit_leg_plan_embeddings,
    gin_encoder_default_kwargs,
    run_mean_baseline,
    fit_standard_regressors,
    run_mscn,
    evaluate_runtime_metrics,
)

set_seed(SEED)


## Config

In [ ]:
import json

SCHEMA_NAME = "tpcds"
LAKEHOUSE_INSTANCE_NAME = "lakehouse-h"
TEST_FRAC = 0.2


def queries_dir_for(run_id: str) -> str:
    return f"/mnt/lakehouse-raw-results/{SCHEMA_NAME}/{LAKEHOUSE_INSTANCE_NAME}/{run_id}/queries"


_tpcds_1200_report = json.loads((ROOT / "Workloads" / "tpcds_1200" / "generation_report.json").read_text())
TPCDS_1200_FIXED_TEST_QIDS = [Path(f).stem for f in _tpcds_1200_report["train_test_split"]["test_query_files"]]

LEGS = {
    "tpcds": dict(
        collection="tpcds_1200",
        run_ids=["20260816-120034Z"],
        vendi=22.72,
        fixed_test_qids=TPCDS_1200_FIXED_TEST_QIDS,
    ),
    "sqlbarber": dict(
        collection="sqlbarber_1000_tokens_tpcds",
        run_ids=["20260818-033107Z"],
        vendi=16.38,
    ),
    "bootstrapping_lcm": dict(
        collection="bootstrapping_lcm_1000_tokens_tpcds",
        run_ids=["20260816-171633Z"],
        vendi=18.98,
    ),
    "sql_factory": dict(
        collection="sql_factory_1000_tokens_tpcds",
        run_ids=["20260817-133213Z"],
        vendi=52.44,
    ),
    "e2etune": dict(
        collection="e2etune_1000_tokens_tpcds",
        run_ids=["20260817-044319Z"],
        vendi=55.32,
    ),
    "sqlstorm": dict(
        collection="sqlstorm_1000_tokens_tpcds_new",
        run_ids=["20260819-161625Z"],
        vendi=77.64,
    ),
    "gpt_generated": dict(
        collection="gpt_generated_1000_tpcds_tokens",
        run_ids=["20260813-214605Z"],
        vendi=106.06,
    ),
}

LEG_COLORS = {
    "tpcds": "#9ca3af",
    "sqlbarber": "#a855f7",
    "bootstrapping_lcm": "#6366f1",
    "sql_factory": "#0891b2",
    "e2etune": "#059669",
    "sqlstorm": "#db2777",
    "gpt_generated": "#ea580c",
}

OURS = "gpt_generated"

print(f"{len(LEGS)} legs configured; Vendi range "
      f"{min(l['vendi'] for l in LEGS.values())}-{max(l['vendi'] for l in LEGS.values())}")


In [ ]:
mixed, AVAILABLE_LEGS, SKIPPED_LEGS = build_mixed_bundle_lenient(
    LEGS,
    schema=SCHEMA_NAME,
    instance=LAKEHOUSE_INSTANCE_NAME,
    metric=METRIC,
    xcol=XCOL,
    ycol=YCOL,
    parsed_results_root=PARSED_RESULTS_ROOT,
    seed=SEED,
    test_frac=TEST_FRAC,
    use_scalar_runtime=True,
    queries_dir_for=queries_dir_for,
)

AVAILABLE_LEGS = sorted(AVAILABLE_LEGS, key=lambda name: LEGS[name]["vendi"])

print()
print(f"available legs ({len(AVAILABLE_LEGS)}/{len(LEGS)}):",
      [f"{n} (Vendi {LEGS[n]['vendi']})" for n in AVAILABLE_LEGS])
print(f"skipped legs ({len(SKIPPED_LEGS)}/{len(LEGS)}):", SKIPPED_LEGS)
print(f"mixed test total: {len(mixed.test_qids)}")

LEG_TRAIN_QIDS = {name: [q for q in mixed.train_qids if q.startswith(f"{name}::")] for name in AVAILABLE_LEGS}
LEG_TEST_QIDS = {name: [q for q in mixed.test_qids if q.startswith(f"{name}::")] for name in AVAILABLE_LEGS}

for name in AVAILABLE_LEGS:
    assert not (set(LEG_TRAIN_QIDS[name]) & set(LEG_TEST_QIDS[name])), f"{name}: train/test leakage!"
    print(f"{name}: n_train={len(LEG_TRAIN_QIDS[name])}  n_test_own_half={len(LEG_TEST_QIDS[name])}")
print("leakage check passed: no leg trains on its own held-out queries")


## Part 1a: plan encoder choice

In [ ]:
ENCODER_CLS = TrinoGraphGINPlanEncoder

_probe_kwargs = gin_encoder_default_kwargs(seed=SEED)
print(f"encoder: {ENCODER_CLS.__name__}")
print(f"device={_probe_kwargs['device']}  epochs={_probe_kwargs['epochs']}  "
      f"hidden={_probe_kwargs['hidden']}  n_layers={_probe_kwargs['n_layers']}  "
      f"emb_dim={_probe_kwargs['emb_dim']}")
print(f"fixed_vocab size={len(_probe_kwargs['fixed_vocab'])} (+UNK) -- "
      f"the schema's full reachable-operator set, shared identically across every leg")
print("fit once per (leg, seed) inside Part 1's sweep loop below, not here -- see this Part's markdown for why.")


## Part 1b: the offline size sweep


In [ ]:
CHECKPOINTS = [5, 10, 20, 50, 100, 200, 250, 500, 10**9] #(10**9 always clips to each leg's true max)
N_SEEDS = 5
KEPT_STANDARD = ["random_forest", "xgboost", "lightgbm", "mlp", "gp_rbf"]
KEPT_DEEP = ["mscn"]
KEPT_MODEL_FAMILIES = ["mean"] + KEPT_STANDARD + KEPT_DEEP
NUM_EPOCHS = 30

def training_cost_this_step(n_cumulative, n_previous_checkpoint):
    return n_cumulative


_DEEP_RUNNERS = {"mscn": run_mscn}


def run_kept_models(leg_data, qids, emb, *, seed):
    results = {"mean": run_mean_baseline(leg_data)}

    if KEPT_STANDARD:
        Z_ckpt = filter_embeddings_by_qids(emb["Z_train_full"], qids)
        X_train, y_train, _ = build_point_dataset(Z_ckpt, emb["leg_data_full"].runs_train, xcol=XCOL)

        standard_results = fit_standard_regressors(
            X_train, y_train, emb["X_test"], emb["y_test"], emb["kept_test_qids"],
            seed=seed, include_gp="gp_rbf" in KEPT_STANDARD,
            model_names=[name for name in KEPT_STANDARD if name != "gp_rbf"],
        )
        for name in KEPT_STANDARD:
            if name in standard_results:
                results[name] = standard_results[name]

    for name in KEPT_DEEP:
        try:
            results[name] = _DEEP_RUNNERS[name](leg_data, seed=seed, num_epochs=NUM_EPOCHS)
        except Exception as e:
            print(f"[skipping {name}] {type(e).__name__}: {e}")

    return results


print(f"sweep scope: {len(AVAILABLE_LEGS)} legs x {N_SEEDS} seeds x up to {len(CHECKPOINTS)} checkpoints "
      f"x {len(KEPT_MODEL_FAMILIES)} models (checkpoints beyond a leg's own training-set size collapse "
      f"into a single 'full' point, deduplicated by nested_query_subsamples; individual model failures "
      f"at a given checkpoint are skipped, not fatal)")


In [ ]:
rows_results = []
rows_stats = []
rows_train_diagnostics = []
rows_training_history = []

for leg in AVAILABLE_LEGS:
    train_qids_full = LEG_TRAIN_QIDS[leg]
    other_legs = [o for o in AVAILABLE_LEGS if o != leg]

    for seed_idx in range(N_SEEDS):
        seed = SEED + seed_idx
        emb = fit_leg_plan_embeddings(
            mixed, train_qids_full,
            encoder_cls=ENCODER_CLS, seed=seed, xcol=XCOL,
        )

        subsamples = nested_query_subsamples(train_qids_full, CHECKPOINTS, seed=seed)
        prev_n = 0

        for n, qids in subsamples.items():
            print(f"=== [offline] leg={leg} seed={seed} n={n} (of {len(train_qids_full)}) ===")

            leg_data = subset_bundle(mixed, train_qids=qids)

            stats = training_subset_stats(mixed.plans_by_query, mixed.runs_by_query, qids, xcol=XCOL)
            rows_stats.append({"train_leg": leg, "seed": str(seed), "n_queries": n, **stats})

            results = run_kept_models(leg_data, qids, emb, seed=seed)
            cost = training_cost_this_step(n, prev_n)
            prev_n = n

            for model_name, res in results.items():
                degenerate = bool(res.get("degenerate_fit", False))

                # train-set fit -- one row per (leg, seed, n, model), independent of
                # which OTHER leg's held-out slice is being scored below
                if res.get("train_metrics"):
                    rows_train_diagnostics.append({
                        "train_leg": leg, "seed": str(seed), "n_queries": n, "model": model_name,
                        "degenerate_fit": degenerate,
                        **{f"train_{k}": v for k, v in res["train_metrics"].items()},
                    })

                # per-round (xgboost) / per-epoch (mscn) training curve, if this
                # model family produced one (random_forest/gp_rbf/mean do not)
                for step_row in res.get("history") or []:
                    rows_training_history.append({
                        "train_leg": leg, "seed": str(seed), "n_queries": n, "model": model_name,
                        **step_row,
                    })

                if "kept_test_qids" not in res:
                    print(f"[skipping] {model_name} (leg={leg}, n={n}): no kept_test_qids in its result")
                    continue
                kept = res["kept_test_qids"]
                preds = np.asarray(res["predictions"]["preds"])
                targets = np.asarray(res["predictions"]["targets"])

                for other in other_legs:
                    prefix = f"{other}::"
                    idx = [i for i, q in enumerate(kept) if q.startswith(prefix)]
                    if not idx:
                        continue
                    m = evaluate_runtime_metrics(preds[idx], targets[idx])
                    rows_results.append({
                        "train_leg": leg, "seed": str(seed), "n_queries": n,
                        "model": model_name, "test_leg": other, "n": len(idx),
                        "training_cost_this_step": cost, "degenerate_fit": degenerate, **m,
                    })

df_results_all = pd.DataFrame(rows_results)
df_stats_all = pd.DataFrame(rows_stats)
df_train_diagnostics_all = pd.DataFrame(rows_train_diagnostics)
df_training_history_all = pd.DataFrame(rows_training_history)
print(f"\nsweep done: {len(df_results_all)} result rows, {len(df_stats_all)} subset-stat rows, "
      f"{len(df_train_diagnostics_all)} train-diagnostic rows, {len(df_training_history_all)} training-history rows")
if "degenerate_fit" in df_results_all.columns and df_results_all["degenerate_fit"].any():
    n_deg = df_results_all.loc[df_results_all["degenerate_fit"], ["train_leg", "seed", "n_queries", "model"]].drop_duplicates()
    print(f"NOTE: {len(n_deg)} (leg, seed, n, model) cells flagged degenerate_fit=True -- see Notes for how to treat these.")


## Part 2: save results

Same `results/<name>_<timestamp>.csv` convention as every other notebook here, scoped to
this experiment's own `cold_start/results/` folder, with an `offline` tag so this sweep's
files never collide with the online notebook's.


In [ ]:
import datetime as dt

SAVE_EVERYTHING = False

out_dir = Path.cwd() / "results"
out_dir.mkdir(exist_ok=True)
stamp = f"{dt.datetime.utcnow():%Y%m%dT%H%M%SZ}"

if SAVE_EVERYTHING:
    stats_path = out_dir / f"cold_start_dose_response_offline_subset_stats_{stamp}.csv"
    df_stats_all.to_csv(stats_path, index=False)
    print("saved:", stats_path)
    
    train_diag_path = out_dir / f"cold_start_dose_response_offline_train_diagnostics_{stamp}.csv"
    df_train_diagnostics_all.to_csv(train_diag_path, index=False)
    print("saved:", train_diag_path)
    
    history_path = out_dir / f"cold_start_dose_response_offline_training_history_{stamp}.csv"
    df_training_history_all.to_csv(history_path, index=False)
    print("saved:", history_path)

results_path = out_dir / f"cold_start_dose_response_offline_qpp_results_{stamp}.csv"
df_results_all.to_csv(results_path, index=False)
print("saved:", results_path)

legs_out_path = out_dir / f"cold_start_dose_response_offline_legs_{stamp}.json"
legs_out_path.write_text(json.dumps({
    "regime": "offline",
    "available": AVAILABLE_LEGS,
    "skipped": SKIPPED_LEGS,
    "vendi": {n: LEGS[n]["vendi"] for n in LEGS},
    "checkpoints": CHECKPOINTS,
    "n_seeds": N_SEEDS,
    "seed_base": SEED,
    "kept_model_families": KEPT_MODEL_FAMILIES,
    "plan_encoder": ENCODER_CLS.__name__,
}, indent=2))
print("saved:", legs_out_path)
